# Fine-tune GPT-2 Vietnamese for Math Word Problems — **V4**

**Core idea:** teacher-KD compact reasoning + PEFT LoRA.

This notebook is intentionally self-contained for Kaggle:

1. pip-install runtime dependencies;
2. load `train.json`, `valid.json`, optional `test.json`, base GPT-2, and required
   `train_kd_compact.jsonl`;
3. validate KD rows against `train.json`;
4. train LoRA stage 1 on answer-only targets;
5. continue LoRA stage 2 on KD compact-reasoning targets with answer-only replay;
6. generate validation/test outputs and reports.

The hidden-test concern is now central: the updated `valid.json` has little/no
query overlap with train, so V4 optimizes for compact, verified supervision
rather than memorizing repeated valid queries.


In [ ]:
# ============================================================
# 0. Install notebook dependencies
# ============================================================
# The official notebook is expected to run in one Kaggle session. The user has
# verified that pip install works for these packages in Kaggle.
import sys, subprocess, importlib.util

PIP_INSTALL_DEPS = True
PIP_PACKAGES = ["peft", "trl", "accelerate", "datasets", "evaluate"]

if PIP_INSTALL_DEPS:
    print("[pip] Installing:", " ".join(PIP_PACKAGES))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES])
else:
    missing = [p for p in PIP_PACKAGES if importlib.util.find_spec(p) is None]
    if missing:
        raise ImportError(f"Missing packages and PIP_INSTALL_DEPS=False: {missing}")


In [ ]:
import os, sys, json, math, time, re, random, hashlib, inspect, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
import peft

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("PEFT :", getattr(peft, "__version__", "unknown"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

def find_named_file(filename: str, extra_roots: list[Path] | None = None) -> Path | None:
    env_key = filename.upper().replace(".", "_").replace("-", "_")
    env_path = os.environ.get(env_key) or os.environ.get("KD_FILE")
    if env_path and Path(env_path).exists():
        return Path(env_path)

    roots = []
    if extra_roots:
        roots.extend(extra_roots)
    roots.extend([
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("dataset"),
        Path("."),
    ])
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        if root.name == "input" or str(root).replace("\\", "/").endswith("/kaggle/input"):
            hits = sorted(root.rglob(filename))
            if hits:
                return hits[0]
    return None

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    str(Path("dataset")),                 # local fallback
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    str(Path("GPT2_vietnamese")),         # local fallback
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"      # Phase 2

# Required teacher-KD compact reasoning artifact.
# Default official flow: generate this file outside the offline submission run,
# upload it as a Kaggle input, then leave CREATE_KD_ARTIFACT=False.
# Optional preprocessing flow: set CREATE_KD_ARTIFACT=True in a non-official run
# with OPENAI_API_KEY available; the notebook will create the JSONL before train.
REQUIRE_KD_FILE = True
CREATE_KD_ARTIFACT = False
KD_FILENAME = "train_kd_compact.jsonl"
KD_GENERATION_MODEL = "gpt-4.1"     # cheaper alternative: "gpt-4.1-mini"
KD_GENERATION_MAX_RECORDS = None    # set small number for smoke tests
KD_GENERATION_WORKERS = 4
KD_GENERATION_MAX_OUTPUT_TOKENS = 220
KD_FILE = find_named_file(KD_FILENAME, extra_roots=[Path(DATA_DIR)])
if REQUIRE_KD_FILE and KD_FILE is None and not CREATE_KD_ARTIFACT:
    raise FileNotFoundError(
        f"Cannot find required {KD_FILENAME}. Upload it as a Kaggle input or place it next to train.json."
    )

# Run mode: "phase1" -> infer on valid;  "phase2" -> infer on test
RUN_MODE = "phase1"

# Prompt & special tokens
PROMPT_TEMPLATE = "Bài toán: {q}\nLời giải: "
ANSWER_SUFFIX_TEMPLATE = "\nĐáp án là: {a}"
SAFE_EOS_ID = 50256

# Working dirs
WORKING_DIR               = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
STAGE1_OUTPUT_DIR         = WORKING_DIR / "gpt2_math_lora_v4_stage1"
FINAL_OUTPUT_DIR          = WORKING_DIR / "gpt2_math_lora_v4_final"
STAGE1_VALID_OUTPUT_PATH  = WORKING_DIR / "valid_output_stage1.json"
STAGE1_VALID_REPORT_PATH  = WORKING_DIR / "valid_report_stage1.json"
VALID_OUTPUT_PATH         = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH         = WORKING_DIR / "valid_report.json"
TEST_OUTPUT_PATH          = WORKING_DIR / "test_predictions.json"
VALID_OVERLAP_AUDIT_PATH  = WORKING_DIR / "valid_overlap_audit.json"
KD_COVERAGE_REPORT_PATH   = WORKING_DIR / "kd_coverage_report.json"
ABLATION_DIR              = WORKING_DIR / "ablations"

# Data
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
FILTER_DUPLICATE_QUERIES = False
DROP_EXACT_DUPLICATES = True
DROP_NON_EXTRACTABLE = True

# KD target constraints
MAX_KD_REASON_TOKENS = 96
ANSWER_ONLY_REPLAY_RATIO = 0.25

# Lengths
N_POSITIONS             = 1024
MAX_LENGTH_STAGE1       = 256
MAX_LENGTH_STAGE2       = 320
MAX_NEW_TOKENS_STAGE1   = 32
MAX_NEW_TOKENS          = 96

# Training
STAGE1_EPOCHS            = 1
STAGE2_EPOCHS            = 1.5
PER_DEVICE_BATCH_SIZE    = 4
GRAD_ACCUM               = 8
STAGE1_LR                = 5e-5
STAGE2_LR                = 1e-5
WARMUP_RATIO             = 0.05
WEIGHT_DECAY             = 0.01
SEED                     = 42

# LoRA
LORA_R                   = 16
LORA_ALPHA               = 32
LORA_DROPOUT             = 0.05
LORA_TARGET_MODULES      = ["c_attn", "c_proj", "c_fc"]

# Decoding
DECODE_MODE              = "beam"
NUM_BEAMS                = 2
NO_REPEAT_NGRAM          = 4
REPETITION_PENALTY       = 1.2
LENGTH_PENALTY           = 0.9
RUN_STAGE1_GENERATION_EVAL = True
RUN_DECODING_ABLATIONS   = False
DECODING_ABLATIONS       = [
    ("greedy", "greedy", 1),
    ("beam2", "beam", 2),
    ("beam4", "beam", 4),
]

# self-consistency only
SC_N                      = 5
SC_TEMPERATURE            = 0.7
SC_TOP_P                  = 0.9

# Inference safety knobs
INFER_FP16                = True
USE_TYPE_AWARE_FEWSHOT    = False

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

print("TRAIN_FILE        :", TRAIN_FILE)
print("VALID_FILE        :", VALID_FILE)
print("MODEL_NAME        :", MODEL_NAME)
print("KD_FILE           :", KD_FILE)
print("CREATE_KD_ARTIFACT:", CREATE_KD_ARTIFACT)
print("STAGE1_OUTPUT_DIR :", STAGE1_OUTPUT_DIR)
print("FINAL_OUTPUT_DIR  :", FINAL_OUTPUT_DIR)
print("RUN_MODE          :", RUN_MODE)


In [ ]:
# ============================================================
# 2. Tokenizer smoke-check
# ============================================================
def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # The provided tokenizer maps id 50256 to a normal token string ("hue")
    # even though the task requires using 50256 as EOS/PAD. Strip it manually.
    return tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

probe = "Bài toán: 2+3=?\nLời giải: "
print("Vocab size:", tokenizer.vocab_size)
print("Token ids :", tokenizer(probe, add_special_tokens=False)["input_ids"][:20], "...")
print("SAFE_EOS raw decode:", repr(tokenizer.decode([SAFE_EOS_ID])))
print("SAFE_EOS stripped  :", repr(decode_model_text(tokenizer, [16, SAFE_EOS_ID])))


In [ ]:
# ============================================================
# 3. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
test_records_for_leak_check = load_records(TEST_FILE) if TEST_FILE.exists() else []

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records), "| test_for_leak_check:", len(test_records_for_leak_check))
print("first query:", train_records[0]["query_vi"][:160])
print("first type :", train_records[0].get("type"))


In [ ]:
# ============================================================
# 4. Answer extraction + data cleaning + target normalization
# ============================================================

# ---- regex pool (case-insensitive, full-width tolerant) ----
RE_ANCHORS_VI = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
]
RE_ANCHORS_EN = [
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
RE_BOXED = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

# Numeric literal patterns
RE_NUM_VI_DEC = re.compile(r"-?\d+,\d+")          # 1,5
RE_NUM_EN_DEC = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")
RE_NUM_LOOSE  = re.compile(r"-?\d+(?:[.,]\d+)?")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}

def _clean_tail(s: str) -> str:
    s = s.strip()
    # take first non-empty line after anchor
    s = s.split("\n", 1)[0].strip()
    # strip trailing punctuation / currency
    s = re.sub(r"[.,;:。、,]+$", "", s)
    # drop "đô la", "USD", "%" trailing units
    s = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", s, flags=re.IGNORECASE)
    s = s.strip()
    return s

def extract_anchor_answer(text: str | None) -> str | None:
    if not text:
        return None
    # last hit per anchor wins (final answer is usually at the end)
    best_pos, best_tail = -1, None
    for pat in RE_ANCHORS_VI + RE_ANCHORS_EN:
        for m in pat.finditer(text):
            if m.end() > best_pos:
                best_pos = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    # boxed fallback (take LAST boxed)
    boxes = RE_BOXED.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(s: str | None) -> float | None:
    if s is None:
        return None
    t = s.strip()
    if not t:
        return None
    # pure VI decimal
    if RE_NUM_VI_DEC.fullmatch(t):
        try:
            v = float(t.replace(",", "."))
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    if RE_NUM_EN_DEC.fullmatch(t):
        try:
            v = float(t)
            return v if math.isfinite(v) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")
    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")
    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)
    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    try:
        v = eval(t.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = float(v)
        return v if math.isfinite(v) else None
    return None

def extract_gold(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("response_vi"))
    return s, parse_number(s)

def extract_pred(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("model_output"))
    return s, parse_number(s)



# ---- V4 canonicalization, audits, KD loading, and target builders ----
def canonicalize_answer(gold_num: float | None) -> str | None:
    if gold_num is None:
        return None
    if abs(gold_num - round(gold_num)) < 1e-9:
        return str(int(round(gold_num)))
    return f"{gold_num:g}"

def numbers_close(a: float | None, b: float | None) -> bool:
    if a is None or b is None:
        return False
    return abs(a - b) <= 1e-9 * max(1.0, abs(a), abs(b))

def stable_fraction(source_id: int) -> float:
    # deterministic pseudo-random value in [0, 1)
    h = hashlib.sha256(str(source_id).encode()).hexdigest()
    return int(h[:8], 16) / 0x100000000

def build_answer_only_target(canonical_answer: str) -> str:
    return f"Đáp án là: {canonical_answer}"

def kd_extract_json_object(text: str) -> dict:
    text = str(text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])
    raise ValueError("Teacher output does not contain a JSON object")

def kd_response_to_text(response) -> str:
    output_text = getattr(response, "output_text", None)
    if output_text:
        return output_text
    chunks = []
    for item in getattr(response, "output", []) or []:
        for content in getattr(item, "content", []) or []:
            text = getattr(content, "text", None)
            if text:
                chunks.append(text)
    if chunks:
        return "\n".join(chunks)
    return str(response)

KD_SYSTEM_PROMPT = """Bạn là giáo viên toán. Nhiệm vụ của bạn là rút gọn lời giải
cho một bài toán tiếng Việt thành reasoning ngắn, chính xác, dễ học cho mô hình
GPT-2 nhỏ.

Chỉ trả về JSON object hợp lệ, không markdown, không giải thích thêm.

JSON schema:
{
  "compact_reasoning": "1-3 câu tiếng Việt ngắn, ưu tiên phép tính/equation, không chứa dòng đáp án cuối",
  "final_answer": "đáp án số"
}

Quy tắc:
- Dựa trên câu hỏi, lời giải gốc và đáp án đúng được cung cấp.
- Không thêm kiến thức ngoài.
- Không viết 'Đáp án là:' trong compact_reasoning.
- Không dùng ####.
- final_answer phải bằng đúng đáp án đúng.
"""

def kd_build_user_prompt(rec: dict, canonical_answer: str) -> str:
    return (
        "Hãy tạo compact_reasoning cho record sau.\n\n"
        f"TYPE:\n{rec.get('type')}\n\n"
        f"QUESTION_VI:\n{rec.get('query_vi')}\n\n"
        f"ORIGINAL_RESPONSE_VI:\n{rec.get('response_vi')}\n\n"
        f"CORRECT_FINAL_ANSWER:\n{canonical_answer}\n"
    )

def maybe_create_kd_artifact_in_notebook(train_recs: list[dict]) -> Path | None:
    """Optional non-official preprocessing path.

    Official offline submission should normally keep CREATE_KD_ARTIFACT=False
    and read an uploaded train_kd_compact.jsonl artifact. This function exists
    so the same notebook can also be used in a preprocessing run with Internet/API.
    """
    if not CREATE_KD_ARTIFACT:
        return KD_FILE

    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("CREATE_KD_ARTIFACT=True requires OPENAI_API_KEY.")

    import importlib.util
    import subprocess
    from concurrent.futures import ThreadPoolExecutor, as_completed

    if importlib.util.find_spec("openai") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])

    def process_one(task):
        source_id, rec = task
        from openai import OpenAI
        client = OpenAI()
        gold_str, gold_num = extract_gold(rec)
        canonical = canonicalize_answer(gold_num)
        if canonical is None:
            return None, {"source_id": source_id, "error": "gold_not_numeric", "gold": gold_str}

        response = client.responses.create(
            model=KD_GENERATION_MODEL,
            instructions=KD_SYSTEM_PROMPT,
            input=kd_build_user_prompt(rec, canonical),
            max_output_tokens=KD_GENERATION_MAX_OUTPUT_TOKENS,
            temperature=0.1,
        )
        obj = kd_extract_json_object(kd_response_to_text(response))
        reasoning = normalize_compact_reasoning(obj.get("compact_reasoning", ""))
        final_answer = str(obj.get("final_answer", canonical)).strip()
        if not reasoning:
            return None, {"source_id": source_id, "error": "empty_reasoning"}
        if not numbers_close(parse_number(final_answer), gold_num):
            return None, {
                "source_id": source_id,
                "error": "final_answer_mismatch",
                "gold": canonical,
                "teacher_final_answer": final_answer,
            }
        return {
            "source_id": source_id,
            "query_vi": rec["query_vi"],
            "type": rec.get("type"),
            "gold_answer": canonical,
            "compact_reasoning": reasoning,
            "final_answer": canonical,
            "teacher_model": KD_GENERATION_MODEL,
            "verified": True,
        }, None

    out_path = WORKING_DIR / KD_FILENAME
    err_path = WORKING_DIR / "train_kd_compact.errors.jsonl"
    done = set()
    if out_path.exists():
        with out_path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    done.add(json.loads(line).get("source_id"))
                except Exception:
                    pass

    tasks = []
    for source_id, rec in enumerate(train_recs):
        if source_id in done:
            continue
        if KD_GENERATION_MAX_RECORDS is not None and len(tasks) >= KD_GENERATION_MAX_RECORDS:
            break
        tasks.append((source_id, rec))

    print(f"[kd-create] output={out_path} model={KD_GENERATION_MODEL} pending={len(tasks)} already_done={len(done)}")
    ok = bad = 0
    with out_path.open("a", encoding="utf-8") as fout, err_path.open("a", encoding="utf-8") as ferr:
        with ThreadPoolExecutor(max_workers=KD_GENERATION_WORKERS) as ex:
            futures = [ex.submit(process_one, task) for task in tasks]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="teacher-kd"):
                try:
                    row, err = fut.result()
                except Exception as exc:
                    row, err = None, {"error": "teacher_call_failed", "detail": repr(exc)}
                if row is not None:
                    fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                    fout.flush()
                    ok += 1
                else:
                    ferr.write(json.dumps(err, ensure_ascii=False) + "\n")
                    ferr.flush()
                    bad += 1
    print(f"[kd-create] done ok={ok} bad={bad} errors={err_path}")
    return out_path

def normalize_compact_reasoning(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text or "")).strip()
    text = re.sub(r"\s+", " ", text)
    # The notebook appends the final answer itself. Strip common answer tails.
    text = re.sub(
        r"(\s*(####\s*[-\d., ]*|"
        r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?[^\n]*|"
        r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?[^\n]*|"
        r"the\s*answer\s*is\s*[:：]?[^\n]*))+\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text

def save_valid_overlap_audit(train_recs: list[dict], valid_recs: list[dict], path: Path):
    train_q = set((r.get("query_vi") or "").strip() for r in train_recs)
    by_type = Counter()
    seen_by_type = Counter()
    seen_ids = []
    for i, rec in enumerate(valid_recs):
        t = rec.get("type") or "UNK"
        by_type[t] += 1
        q = (rec.get("query_vi") or "").strip()
        if q in train_q:
            seen_by_type[t] += 1
            seen_ids.append(i)
    payload = {
        "train_n": len(train_recs),
        "valid_n": len(valid_recs),
        "train_unique_queries": len(train_q),
        "valid_seen_query": len(seen_ids),
        "valid_seen_query_pct": len(seen_ids) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_ids_first20": seen_ids[:20],
        "valid_by_type": dict(sorted(by_type.items())),
        "seen_by_type": dict(sorted(seen_by_type.items())),
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[overlap]", json.dumps(payload, ensure_ascii=False, indent=2))
    return payload

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen = set()
    out = []
    dropped_no_ans = 0
    dropped_dup = 0

    for source_id, rec in enumerate(records):
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_no_ans += 1
            continue

        if split == "train":
            if FILTER_DUPLICATE_QUERIES:
                key = q
            elif DROP_EXACT_DUPLICATES:
                key = (q, r)
            else:
                key = None
            if key is not None:
                if key in seen:
                    dropped_dup += 1
                    continue
                seen.add(key)

        gold_str, gold_num = extract_gold(rec)
        canonical_answer = canonicalize_answer(gold_num)
        if DROP_NON_EXTRACTABLE and split == "train" and canonical_answer is None:
            dropped_no_ans += 1
            continue

        out.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
            "_source_id": source_id,
            "_gold_num": gold_num,
            "_gold_str": gold_str,
            "_canonical_answer": canonical_answer,
        })

    print(f"[{split}] kept={len(out)} dropped_dup={dropped_dup} dropped_no_ans={dropped_no_ans}")
    return out

def load_and_validate_kd(
    kd_path: Path,
    train_recs: list[dict],
    valid_recs: list[dict],
    test_recs: list[dict],
    tokenizer,
) -> tuple[dict[int, dict], dict]:
    valid_q = set((r.get("query_vi") or "").strip() for r in valid_recs)
    test_q = set((r.get("query_vi") or "").strip() for r in test_recs)
    leak_q = valid_q | test_q

    kd_by_source = {}
    invalid = Counter()
    total = 0
    duplicates = 0
    token_lens = []

    with Path(kd_path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                row = json.loads(line)
            except Exception:
                invalid["bad_json"] += 1
                continue

            source_id = row.get("source_id")
            if not isinstance(source_id, int):
                invalid["bad_source_id"] += 1
                continue
            if source_id < 0 or source_id >= len(train_recs):
                invalid["source_id_out_of_range"] += 1
                continue

            train_row = train_recs[source_id]
            q = (row.get("query_vi") or "").strip()
            train_q = (train_row.get("query_vi") or "").strip()
            if q != train_q:
                invalid["query_mismatch"] += 1
                continue
            if q in leak_q:
                invalid["valid_or_test_query_leak"] += 1
                continue
            if row.get("verified") is not True:
                invalid["not_verified"] += 1
                continue

            gold_str, gold_num = extract_gold(train_row)
            canonical_answer = canonicalize_answer(gold_num)
            if canonical_answer is None:
                invalid["train_gold_not_numeric"] += 1
                continue

            final_answer = row.get("final_answer")
            final_num = parse_number(str(final_answer)) if final_answer is not None else None
            if not numbers_close(final_num, gold_num):
                invalid["final_answer_mismatch"] += 1
                continue

            reasoning = normalize_compact_reasoning(row.get("compact_reasoning"))
            if not reasoning:
                invalid["empty_reasoning"] += 1
                continue
            n_reason_tokens = len(tokenizer(reasoning, add_special_tokens=False)["input_ids"])
            if n_reason_tokens > MAX_KD_REASON_TOKENS:
                invalid["reasoning_too_long"] += 1
                continue

            if source_id in kd_by_source:
                duplicates += 1
                invalid["duplicate_source_id"] += 1
                continue

            kd_by_source[source_id] = {
                "source_id": source_id,
                "compact_reasoning": reasoning,
                "canonical_answer": canonical_answer,
                "teacher_model": row.get("teacher_model"),
                "reason_tokens": n_reason_tokens,
                "line_no": line_no,
            }
            token_lens.append(n_reason_tokens)

    by_type_total = Counter()
    by_type_valid = Counter()
    for i, rec in enumerate(train_recs):
        t = rec.get("type") or "UNK"
        by_type_total[t] += 1
        if i in kd_by_source:
            by_type_valid[t] += 1

    report = {
        "kd_path": str(kd_path),
        "total_kd_lines": total,
        "valid_kd_rows": len(kd_by_source),
        "invalid_counts": dict(sorted(invalid.items())),
        "duplicate_valid_source_ids": duplicates,
        "coverage_over_raw_train": len(kd_by_source) / len(train_recs) if train_recs else 0.0,
        "reason_token_stats": {
            "min": min(token_lens) if token_lens else None,
            "max": max(token_lens) if token_lens else None,
            "mean": sum(token_lens) / len(token_lens) if token_lens else None,
        },
        "coverage_by_type": {
            t: {
                "train_n": by_type_total[t],
                "valid_kd": by_type_valid[t],
                "coverage": by_type_valid[t] / by_type_total[t] if by_type_total[t] else 0.0,
            }
            for t in sorted(by_type_total)
        },
    }
    KD_COVERAGE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[kd]", json.dumps({
        "total_kd_lines": report["total_kd_lines"],
        "valid_kd_rows": report["valid_kd_rows"],
        "invalid_counts": report["invalid_counts"],
        "coverage_over_raw_train": report["coverage_over_raw_train"],
        "reason_token_stats": report["reason_token_stats"],
    }, ensure_ascii=False, indent=2))
    if REQUIRE_KD_FILE and len(kd_by_source) == 0:
        raise ValueError("KD file exists, but no valid KD rows survived validation.")
    return kd_by_source, report

def build_stage1_records(records: list[dict]) -> list[dict]:
    out = []
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is None:
            continue
        out.append({**rec, "response_vi": build_answer_only_target(canonical), "_stage": "stage1_answer_only"})
    print(f"[build:stage1] {len(out)}")
    return out

def build_stage2_records(records: list[dict], kd_by_source: dict[int, dict]) -> list[dict]:
    out = []
    kd_used = 0
    fallback = 0
    replay = 0
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is None:
            continue
        source_id = rec["_source_id"]
        kd = kd_by_source.get(source_id)
        if kd is not None:
            target = f"{kd['compact_reasoning']}\nĐáp án là: {canonical}"
            kd_used += 1
        else:
            target = build_answer_only_target(canonical)
            fallback += 1
        out.append({**rec, "response_vi": target, "_stage": "stage2_kd_or_fallback", "_has_kd": kd is not None})

        if stable_fraction(source_id) < ANSWER_ONLY_REPLAY_RATIO:
            out.append({**rec, "response_vi": build_answer_only_target(canonical), "_stage": "stage2_answer_replay", "_has_kd": kd is not None})
            replay += 1
    print(f"[build:stage2] total={len(out)} kd_used={kd_used} fallback={fallback} replay={replay}")
    return out

KD_FILE = maybe_create_kd_artifact_in_notebook(train_records)
if REQUIRE_KD_FILE and KD_FILE is None:
    raise FileNotFoundError(f"Cannot find or create required {KD_FILENAME}")

valid_overlap_audit = save_valid_overlap_audit(train_records, valid_records, VALID_OVERLAP_AUDIT_PATH)
train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid")

kd_by_source, kd_report = load_and_validate_kd(
    KD_FILE, train_records, valid_records, test_records_for_leak_check, tokenizer
)

train_stage1 = build_stage1_records(train_clean)
valid_stage1 = build_stage1_records(valid_clean)
train_stage2 = build_stage2_records(train_clean, kd_by_source)
# valid has no teacher KD; eval loss is answer-focused to avoid validating on leaked gold reasoning.
valid_stage2 = build_stage1_records(valid_clean)

print("\nExample targets:")
print("[stage1]", train_stage1[0]["response_vi"])
print("[stage2]", train_stage2[0]["response_vi"][:800])


In [ ]:
# ============================================================
# 5. SFT dataset (anchor-safe truncation) + collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response); mask loss on prompt + padding.

    Truncation policy: if the full sequence exceeds max_length, we drop
    from the *middle* of the response so that the final
    `\nĐáp án là: <num>\u200b<eos>` always survives.
    """

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"])
        response = rec["response_vi"]

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        budget = self.max_length - len(p_ids)
        if budget <= 4:
            # prompt itself too long → truncate from left of prompt
            p_ids = p_ids[-(self.max_length - 8):]
            budget = self.max_length - len(p_ids)

        if len(r_ids) > budget:
            # keep the last `tail_keep` tokens (carries the anchor + answer + EOS)
            tail_keep = min(96, budget // 2)
            head_keep = budget - tail_keep
            r_ids = r_ids[:head_keep] + r_ids[-tail_keep:]

        ids    = p_ids + r_ids
        labels = [-100] * len(p_ids) + r_ids
        # Defensive clamp: never feed out-of-range ids.
        ids    = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

# quick sanity check
_ds_probe_stage1 = SFTDataset(train_stage1[:1], tokenizer, MAX_LENGTH_STAGE1)
_ds_probe_stage2 = SFTDataset(train_stage2[:1], tokenizer, MAX_LENGTH_STAGE2)
_sample_stage1 = _ds_probe_stage1[0]
_sample_stage2 = _ds_probe_stage2[0]
print("stage1 sample len:", len(_sample_stage1["input_ids"]),
      "| n_loss_tokens:", sum(1 for x in _sample_stage1["labels"] if x != -100))
print("stage1 tail:", decode_model_text(tokenizer, _sample_stage1["input_ids"][-15:]))
print("stage2 sample len:", len(_sample_stage2["input_ids"]),
      "| n_loss_tokens:", sum(1 for x in _sample_stage2["labels"] if x != -100))
print("stage2 tail:", decode_model_text(tokenizer, _sample_stage2["input_ids"][-30:]))


In [ ]:
# ============================================================
# 6. Evaluation utilities (uses the upgraded extractors above)
# ============================================================
def rel_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(re_val, is_extractable):
    if not is_extractable or re_val is None:
        return 0
    if re_val <= 0.01: return 10
    if re_val <= 0.10: return 5
    if re_val <= 0.50: return 1
    return 0

def evaluate(pred_items, gold_items):
    assert len(pred_items) == len(gold_items), (len(pred_items), len(gold_items))
    rows = []
    total = 0
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in zip(pred_items, gold_items):
        gold_str, gold_num = extract_gold(gold_rec)
        pred_str, pred_num = extract_pred(pred_rec)

        is_extractable = pred_str is not None
        extractable += int(is_extractable)
        re_val = rel_error(pred_num, gold_num)
        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s
        buckets[s] = buckets.get(s, 0) + 1

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_str, "gold_num": gold_num,
            "pred_answer": pred_str, "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": n * 10,
            "score_10": total / n if n else 0.0,
            "score_pct": (total / (n * 10)) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }

def save_evaluation_report(pred_path, gold_records, report_path):
    pred_path = Path(pred_path); report_path = Path(report_path)
    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)
    result = evaluate(pred_items, gold_records)
    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Wrote {report_path}")
    return result


In [ ]:
# ============================================================
# 7. StoppingCriteria + decoding
# ============================================================
class StopOnAnswerLine(StoppingCriteria):
    """Stop when generation has emitted a final `Đáp án là: <number>` AND the
    next token starts a new line or EOS. Cheap heuristic: stop when we have
    decoded a substring matching the answer-line regex with a trailing newline
    or when EOS appears.
    """
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID,
                 patience_tokens: int = 24):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None     # token index when first matched

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        # decode only the generated tail (cheaper)
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 6:
            return False
        text = self.tok.decode(gen_tail, skip_special_tokens=True)
        m = self.re_answer.search(text)
        if m:
            if self._matched_at is None:
                self._matched_at = gen_tail.numel()
            # let it spit out a few more digits, then stop on newline or patience
            if "\n" in text[m.end():]:
                return True
            if gen_tail.numel() - self._matched_at >= self.patience:
                return True
        return False


def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())


# ---- optional type-aware few-shot exemplars ----
FEWSHOT_BY_TYPE = {
    "GSM_AnsAug": (
        "Bài toán: Lan có 5 quả cam, mẹ cho thêm 3 quả. Hỏi Lan có bao nhiêu quả cam?\n"
        "Lời giải: Lan ban đầu có 5 quả. Mẹ cho thêm 3 quả. Tổng cộng 5 + 3 = 8 quả.\n"
        "Đáp án là: 8\n\n"
    ),
    "GSM_Rephrased": (
        "Bài toán: Nếu Nam có 10 viên kẹo và ăn 4 viên thì còn bao nhiêu?\n"
        "Lời giải: Nam ăn 4 trên 10 viên kẹo nên còn 10 - 4 = 6 viên.\n"
        "Đáp án là: 6\n\n"
    ),
    "MATH_AnsAug": (
        "Bài toán: Tính giá trị của $2^5$.\n"
        "Lời giải: $2^5 = 2 \\cdot 2 \\cdot 2 \\cdot 2 \\cdot 2 = 32$.\n"
        "Đáp án là: 32\n\n"
    ),
    "MATH_Rephrased": (
        "Bài toán: Tìm $x$ thỏa mãn $3x = 12$.\n"
        "Lời giải: Chia hai vế cho 3, ta có $x = 12/3 = 4$.\n"
        "Đáp án là: 4\n\n"
    ),
    "GSM_FOBAR": (
        "Bài toán: An có x quả bóng. An cho 2 quả. Còn lại 5 quả. "
        "Nếu biết câu trả lời là 5 thì x bằng bao nhiêu?\n"
        "Lời giải: x - 2 = 5 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "GSM_SV": (
        "Bài toán: Bình có x cái bút. Bình cho bạn 3 cái, còn 4 cái. Tìm x.\n"
        "Lời giải: x - 3 = 4 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "MATH_FOBAR": (
        "Bài toán: $f(x) = 2x + X$. Nếu $f(3) = 7$ thì $X$ bằng bao nhiêu?\n"
        "Lời giải: $2 \\cdot 3 + X = 7$ nên $X = 1$.\n"
        "Đáp án là: 1\n\n"
    ),
    "MATH_SV": (
        "Bài toán: $X + 2 = 5$. Tìm $X$.\n"
        "Lời giải: $X = 5 - 2 = 3$.\n"
        "Đáp án là: 3\n\n"
    ),
}

def build_prompt_with_fewshot(rec: dict) -> str:
    if not USE_TYPE_AWARE_FEWSHOT:
        return build_prompt(rec)
    fs = FEWSHOT_BY_TYPE.get(rec.get("type"), "")
    return fs + build_prompt(rec)


def _vote_answer(cands: list[str | None]) -> str | None:
    nums = []
    for c in cands:
        n = parse_number(extract_anchor_answer(c))
        if n is not None:
            nums.append((round(n, 6), c))
    if not nums:
        return cands[0] if cands else None
    counter = Counter(n for n, _ in nums)
    top_num, _ = counter.most_common(1)[0]
    for n, c in nums:
        if n == top_num:
            return c
    return cands[0]



def has_peft_adapter(path_like) -> bool:
    p = Path(path_like)
    return p.exists() and (p / "adapter_config.json").exists()

def load_tokenizer_for_generation(model_path_or_name):
    p = Path(model_path_or_name)
    tok_source = model_path_or_name if p.exists() and (p / "tokenizer_config.json").exists() else MODEL_NAME
    tok = AutoTokenizer.from_pretrained(tok_source, local_files_only=True)
    tok.pad_token_id = SAFE_EOS_ID
    tok.eos_token_id = SAFE_EOS_ID
    if tok.padding_side != "left":
        tok.padding_side = "left"
    return tok

def load_model_for_generation(model_path_or_name, device, dtype):
    if has_peft_adapter(model_path_or_name):
        print(f"[infer] Loading base model + PEFT adapter: {model_path_or_name}")
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=dtype, local_files_only=True,
        )
        base.config.pad_token_id = SAFE_EOS_ID
        base.config.eos_token_id = SAFE_EOS_ID
        model = PeftModel.from_pretrained(base, str(model_path_or_name), local_files_only=True)
        try:
            model = model.merge_and_unload()
            print("[infer] Merged LoRA adapter into base model for faster generation.")
        except Exception as exc:
            print("[infer] Could not merge adapter; using PEFT wrapper:", repr(exc))
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_path_or_name, torch_dtype=dtype, local_files_only=True,
        )
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    return model.to(device)


@torch.inference_mode()
def generate_outputs(model_path_or_name, records, output_path,
                     max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
                     num_beams_override=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[infer] Loading {model_path_or_name} on {device} (mode={mode}) ...", flush=True)

    tok = load_tokenizer_for_generation(model_path_or_name)

    dtype = torch.float16 if (device == "cuda" and INFER_FP16) else torch.float32
    model = load_model_for_generation(model_path_or_name, device=device, dtype=dtype)
    model.eval()
    vocab_n = model.get_input_embeddings().weight.shape[0]
    # GPT-2 absolute positional embedding hard limit. Crossing this triggers a
    # CUDA assert (`indexSelectSmallIndex: srcIndex < srcSelectDimSize`) inside
    # the position-embedding lookup. We MUST keep prompt_len + new_tokens <= n_pos.
    n_pos = int(getattr(model.config, "n_positions",
                        getattr(model.config, "max_position_embeddings", 1024)))

    outputs, t0 = [], time.time()
    for idx, rec in enumerate(tqdm(records, desc=f"gen[{mode}]")):
        prompt = build_prompt_with_fewshot(rec)

        # ---- Budget-aware tokenization ----
        # Leave room for `max_new_tokens` new tokens inside the 1024 position cap.
        # If the prompt is too long, truncate from the LEFT so the trailing
        # "Lời giải:" anchor is preserved (the answer cue MUST stay at the end).
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        # safety clamp on token ids (tokenizer.vocab may > model embedding)
        full_ids = [min(t, vocab_n - 1) for t in full_ids]
        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]

        # Final safety clamp: never let new_tokens push past n_pos.
        eff_new = max(8, min(max_new_tokens, n_pos - prompt_len))

        common_kwargs = dict(
            input_ids=ids, attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID, eos_token_id=SAFE_EOS_ID,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            repetition_penalty=REPETITION_PENALTY,
            stopping_criteria=StoppingCriteriaList([
                StopOnAnswerLine(tok, prompt_len=prompt_len)
            ]),
        )

        if mode == "greedy":
            gen = model.generate(do_sample=False, num_beams=1, **common_kwargs)
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "beam":
            beam_count = num_beams_override or NUM_BEAMS
            gen = model.generate(
                do_sample=False, num_beams=beam_count,
                length_penalty=LENGTH_PENALTY, early_stopping=True,
                **common_kwargs,
            )
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "self_consistency":
            cands = []
            for s in range(SC_N):
                gen = model.generate(
                    do_sample=True, temperature=SC_TEMPERATURE, top_p=SC_TOP_P,
                    num_beams=1, **common_kwargs,
                )
                cands.append(decode_model_text(tok, gen[0, prompt_len:]))
            text = _vote_answer(cands) or cands[0]
        else:
            raise ValueError(mode)

        outputs.append({
            "id": idx,
            "query_vi": rec["query_vi"],
            "type": rec.get("type"),
            "model_output": text,
        })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)
    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"[infer] Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    # free
    del model
    torch.cuda.empty_cache()
    return outputs


In [ ]:
# ============================================================
# 8. PEFT LoRA curriculum training
# ============================================================
def build_training_args(output_dir: Path, epochs: float, lr: float):
    ta_kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        ta_kwargs["eval_strategy"] = "epoch"
    else:
        ta_kwargs["evaluation_strategy"] = "epoch"
    return TrainingArguments(**ta_kwargs)

def build_lora_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

def train_lora_stage(
    *,
    stage_name: str,
    model,
    tokenizer,
    train_records_for_stage: list[dict],
    valid_records_for_stage: list[dict],
    output_dir: Path,
    max_length: int,
    epochs: float,
    lr: float,
):
    print("\n" + "=" * 88)
    print(f"[train:{stage_name}] train={len(train_records_for_stage)} valid={len(valid_records_for_stage)} "
          f"max_length={max_length} epochs={epochs} lr={lr}")

    train_ds = SFTDataset(train_records_for_stage, tokenizer, max_length)
    eval_sub = min(200, len(valid_records_for_stage))
    valid_ds = SFTDataset(valid_records_for_stage[:eval_sub], tokenizer, max_length)
    collator = PadCollator(pad_id=SAFE_EOS_ID)

    eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
    steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
    print(f"[train:{stage_name}] per_device_bs={PER_DEVICE_BATCH_SIZE} grad_accum={GRAD_ACCUM} "
          f"gpus={torch.cuda.device_count()} eff_batch={eff_batch} steps/epoch={steps_per_epoch}")

    trainer = Trainer(
        model=model,
        args=build_training_args(output_dir, epochs, lr),
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
    )

    t0 = time.time()
    trainer.train()
    train_dt = time.time() - t0
    print(f"[train:{stage_name}] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    model_hash = sha256_dir(output_dir)
    (output_dir / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    print(f"[train:{stage_name}] saved adapter -> {output_dir} | SHA256: {model_hash}")

    del trainer
    torch.cuda.empty_cache()
    return model, train_dt

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = build_lora_model()

model, stage1_train_dt = train_lora_stage(
    stage_name="stage1_answer_only_lora",
    model=model,
    tokenizer=tokenizer,
    train_records_for_stage=train_stage1,
    valid_records_for_stage=valid_stage1,
    output_dir=STAGE1_OUTPUT_DIR,
    max_length=MAX_LENGTH_STAGE1,
    epochs=STAGE1_EPOCHS,
    lr=STAGE1_LR,
)

model, stage2_train_dt = train_lora_stage(
    stage_name="stage2_kd_compact_lora",
    model=model,
    tokenizer=tokenizer,
    train_records_for_stage=train_stage2,
    valid_records_for_stage=valid_stage2,
    output_dir=FINAL_OUTPUT_DIR,
    max_length=MAX_LENGTH_STAGE2,
    epochs=STAGE2_EPOCHS,
    lr=STAGE2_LR,
)

print(f"\n[train] total LoRA curriculum wall time: {(stage1_train_dt + stage2_train_dt)/60:.2f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 9. Inference + evaluation
# ============================================================
if RUN_MODE == "phase1":
    if RUN_STAGE1_GENERATION_EVAL:
        stage1_valid_outputs = generate_outputs(
            STAGE1_OUTPUT_DIR,
            valid_records,
            STAGE1_VALID_OUTPUT_PATH,
            max_new_tokens=MAX_NEW_TOKENS_STAGE1,
            mode=DECODE_MODE,
        )
        stage1_valid_result = save_evaluation_report(
            STAGE1_VALID_OUTPUT_PATH,
            valid_records,
            STAGE1_VALID_REPORT_PATH,
        )
        s1 = stage1_valid_result["summary"]
        print("\nStage1 validation score:")
        print(f'{s1["raw_score"]} / {s1["max_raw_score"]}  ({s1["score_pct"]*100:.2f}%)')
        print(f'Score /10: {s1["score_10"]:.4f}')

    valid_outputs = generate_outputs(
        FINAL_OUTPUT_DIR,
        valid_records,
        VALID_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS,
        mode=DECODE_MODE,
    )
    print("\nExample final output:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

    valid_result = save_evaluation_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
    s = valid_result["summary"]
    print("\nFinal validation score:")
    print(f'{s["raw_score"]} / {s["max_raw_score"]}  ({s["score_pct"]*100:.2f}%)')
    print(f'Score /10: {s["score_10"]:.4f}')
    print("Buckets:", s["buckets"])

    if RUN_DECODING_ABLATIONS:
        print("\nRunning same-checkpoint decoding ablations ...")
        ABLATION_DIR.mkdir(exist_ok=True)
        ablation_rows = []
        for ablation_name, ablation_mode, ablation_beams in DECODING_ABLATIONS:
            run_dir = ABLATION_DIR / ablation_name
            run_dir.mkdir(exist_ok=True)
            pred_path = run_dir / "valid_output.json"
            report_path = run_dir / "valid_report.json"
            _ = generate_outputs(
                FINAL_OUTPUT_DIR,
                valid_records,
                pred_path,
                max_new_tokens=MAX_NEW_TOKENS,
                mode=ablation_mode,
                num_beams_override=ablation_beams,
            )
            result = save_evaluation_report(pred_path, valid_records, report_path)
            summary = result["summary"]
            ablation_rows.append({
                "name": ablation_name,
                "mode": ablation_mode,
                "num_beams": ablation_beams,
                **summary,
            })
            print(
                f"[{ablation_name}] raw={summary['raw_score']} "
                f"score10={summary['score_10']:.4f} "
                f"exact10={summary['buckets'][10]} "
                f"extractable={summary['extractable']}"
            )
        (ABLATION_DIR / "ablation_summary.json").write_text(
            json.dumps(ablation_rows, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
else:
    print("Skipping phase1 inference (RUN_MODE != phase1).")


In [ ]:
# ============================================================
# 10. Error analysis preview
# ============================================================
if RUN_MODE == "phase1":
    rows = valid_result["rows"]
    by_type = {}
    for r in rows:
        t = r.get("type") or "UNK"
        by_type.setdefault(t, []).append(r)

    print("Per-type score breakdown:")
    print(f"{'type':<22s} {'n':>4s} {'mean':>6s} {'b10':>4s} {'b5':>4s} {'b1':>4s} {'b0':>4s} {'extr%':>6s}")
    for t, rs in sorted(by_type.items()):
        n = len(rs)
        mean_s = sum(r["score"] for r in rs) / n
        b10 = sum(r["score"] == 10 for r in rs)
        b5  = sum(r["score"] == 5 for r in rs)
        b1  = sum(r["score"] == 1 for r in rs)
        b0  = sum(r["score"] == 0 for r in rs)
        extr = 100 * sum(bool(r["extractable"]) for r in rs) / n
        print(f"{t:<22s} {n:>4d} {mean_s:>6.2f} {b10:>4d} {b5:>4d} {b1:>4d} {b0:>4d} {extr:>5.1f}%")

    bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
    print(f"\n{len(bad)} zero-score samples. Showing first 2:")
    for i, r in bad[:2]:
        pred = valid_outputs[i]; gold = valid_records[i]
        print("=" * 90)
        print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
        print("QUERY:", gold["query_vi"][:400])
        print("GOLD :", gold["response_vi"][-300:])
        print("PRED :", pred["model_output"][:800])


In [ ]:
# ============================================================
# 11. Phase 2 — generate test_predictions.json
# ============================================================
# Set RUN_MODE = "phase2" at the top to activate this block.
if RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"Cannot find test.json at {TEST_FILE}")
    test_records = load_records(TEST_FILE)
    print("test:", len(test_records))

    test_outputs = generate_outputs(
        FINAL_OUTPUT_DIR,
        test_records,
        TEST_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS,
        mode=DECODE_MODE,
    )

    required_keys = {"id", "query_vi", "type", "model_output"}
    missing = [k for k in required_keys if k not in test_outputs[0]]
    if missing:
        raise ValueError(f"Output missing keys: {missing}")

    print("First test prediction:")
    print(json.dumps(test_outputs[0], ensure_ascii=False, indent=2)[:1500])


In [ ]:
# ============================================================
# 12. List saved artifacts
# ============================================================
candidates = [
    STAGE1_VALID_OUTPUT_PATH,
    STAGE1_VALID_REPORT_PATH,
    Path(str(STAGE1_VALID_OUTPUT_PATH) + ".sha256.txt"),
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    VALID_OVERLAP_AUDIT_PATH,
    KD_COVERAGE_REPORT_PATH,
    STAGE1_OUTPUT_DIR / "adapter_config.json",
    STAGE1_OUTPUT_DIR / "adapter_model.safetensors",
    STAGE1_OUTPUT_DIR / "model_hash.txt",
    FINAL_OUTPUT_DIR / "adapter_config.json",
    FINAL_OUTPUT_DIR / "adapter_model.safetensors",
    FINAL_OUTPUT_DIR / "model_hash.txt",
    TEST_OUTPUT_PATH,
    Path(str(TEST_OUTPUT_PATH) + ".sha256.txt"),
]
for p in candidates:
    p = Path(p)
    print(p, "| exists =", p.exists(), "| size =", p.stat().st_size if p.exists() else None)

print("\nWorking dir contents:")
for p in sorted(Path(WORKING_DIR).glob("*")):
    print("-", p)
